In [21]:
import numpy as np
import pandas as pd
import hdbscan

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [22]:
emotion_cols = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval',
    'caring', 'confusion', 'curiosity', 'desire', 'disappointment',
    'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
    'gratitude', 'grief', 'joy', 'love', 'nervousness',
    'optimism', 'pride', 'realization', 'relief', 'remorse',
    'sadness', 'surprise', 'neutral'
]

df = pd.read_parquet("../scaled_emotion_dataset.parquet")

X = df[emotion_cols].values

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5000,
    min_samples=2250,   
    metric='euclidean'
)

labels = clusterer.fit_predict(X)

In [ ]:
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise = np.sum(labels == -1)

print("Clusters :", n_clusters)
print("Noise :", noise)

Clusters : 5
Noise : 37903


In [ ]:
cluster_counts = pd.Series(labels).value_counts().sort_index()

print(cluster_counts)

-1    37903
 0     2408
 1    10033
 2     3698
 3     4818
 4     2219
Name: count, dtype: int64


In [ ]:
mask = labels != -1

if len(np.unique(labels[mask])) > 1:

    sil = silhouette_score(
        X[mask],
        labels[mask]
    )

    db = davies_bouldin_score(
        X[mask],
        labels[mask]
    )

    print(f"Silhouette Score : {sil:.4f}")
    print(f"Davies-Bouldin   : {db:.4f}")

else:
    print("Only one cluster found.")

Silhouette Score : 0.5425
Davies-Bouldin   : 0.8055


In [ ]:
df["Cluster"] = labels

In [ ]:
cluster_profile = df.groupby("Cluster")[emotion_cols].mean()

display(cluster_profile)

,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,disappointment,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
Cluster,,,,,,,,,,,,,,,,,,,,,
-1,-0.119704,0.083870,0.099361,0.171541,0.044201,0.027322,0.123947,0.146556,0.131239,0.123879,...,0.158736,-0.039396,0.175782,0.093998,0.177553,0.159163,0.122282,-0.015101,0.084595,-0.104136
0,-0.469092,-0.100877,-0.110808,-0.166175,-0.453257,-0.388816,-0.096136,-0.201311,-0.212617,-0.282309,...,-0.283968,3.219350,-0.331984,-0.116151,-0.046668,-0.153612,-0.180055,-0.209070,-0.043205,-0.412512
1,1.041031,-0.139687,-0.193723,-0.351490,-0.305664,-0.424672,-0.216819,-0.249645,-0.218048,-0.522695,...,-0.251358,-0.317826,-0.329687,-0.096715,-0.374465,-0.275863,-0.239396,-0.514245,-0.142246,-0.544342
2,-0.468308,-0.136742,-0.089142,-0.174005,-0.501597,-0.409142,-0.204018,-0.235583,-0.209087,1.286458,...,-0.278028,-0.240548,-0.356806,-0.223803,-0.335907,-0.372648,-0.010597,2.643792,-0.148809,-0.329530
3,-0.435472,-0.141586,-0.184759,-0.275848,0.859791,-0.378497,-0.217070,-0.243920,-0.228614,-0.499874,...,-0.290942,-0.315355,-0.321121,-0.223981,-0.170010,-0.334242,-0.269946,-0.510534,-0.154808,2.479687
4,-0.427230,-0.156239,-0.151339,-0.271634,0.087992,3.379015,-0.221194,-0.233923,-0.180273,-0.504877,...,-0.171701,-0.298015,0.140206,-0.182958,-0.360125,0.042053,-0.207141,-0.487499,-0.170815,-0.147250


In [ ]:
print(df["Cluster"].value_counts())

Cluster
-1    37903
 1    10033
 3     4818
 2     3698
 0     2408
 4     2219
Name: count, dtype: int64
